# System-1 Router Baseline — Qwen3.5-0.8B + LoRA + Joint Choice Head

Integrated from a team-tested baseline (CFPB routing) adapted to our
`lets_route` / **xRouteBench** candidate-routing task.

**Task:** given a query + the candidate-model option pool, pick — in one
forward pass — which model should answer. Labels come from the counterfactual
oracle (`router/prepare_dataset.py`): `oracle_target` (cheapest acceptable)
and `acceptable_models` (all within ε of best).

Runs on the **route_f** kernel (Linux CUDA / macOS MPS).

In [11]:
import importlib.util
import importlib.metadata
import sys

required = ['torch', 'transformers', 'peft', 'datasets', 'huggingface_hub', 'tqdm']
missing = [p for p in required if importlib.util.find_spec(p) is None]
print('Kernel:', sys.executable)
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. Select the route_f kernel.')
for p in required:
    print(p, importlib.metadata.version(p))

Kernel: /home/mr/Desktop/Fireworks/.route_f/bin/python
torch 2.15.0
transformers 5.22.1
peft 0.21.2
datasets 5.1.0
huggingface_hub 2.4.0
tqdm 4.70.1


In [12]:
import gc
import json
import math
import platform
import random
import re
import time
import uuid
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F
from huggingface_hub import HfApi
from peft import LoraConfig, PeftModel, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

torch.backends.mha.set_fastpath_enabled(False)

MODEL = 'Qwen/Qwen3.5-0.8B'
MODEL_REVISION = 'main'
SEED = 7

# ── TRAINING HYPERPARAMETERS — KEEP IN SYNC ACROSS CELLS ──
TRAIN_SIZE, TEST_SIZE = 1200, 200   # distinct train / held-out test sampled from splits
STEPS, BATCH = 400, 4                 # 400 steps x batch 4 = 1600 presentations
EVAL_BATCH = 2
RUN_CHAT_BASELINE = True
MAX_STATE_TOKENS, MAX_LENGTH = 128, 768
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE == 'cuda' else torch.float32
HEAD_CONFIG = {'width': 128, 'num_heads': 4, 'joint_layers': 2, 'dropout': 0.1}

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)
assert TRAIN_SIZE >= 9 and TEST_SIZE >= 9 and STEPS > 0 and BATCH > 0 and EVAL_BATCH > 0
assert MAX_STATE_TOKENS > 0 and MAX_LENGTH > MAX_STATE_TOKENS
print(platform.platform())
print('Device:', DEVICE, '| dtype:', DTYPE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(), '| VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 1e9)
print('Selected examples:', TRAIN_SIZE + TEST_SIZE, '| training presentations:', STEPS * BATCH)

def synchronize():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE == 'mps':
        torch.mps.synchronize()

Linux-7.0.0-38-generic-x86_64-with-glibc2.39
Device: cuda | dtype: torch.bfloat16
NVIDIA GeForce RTX 3060 Laptop GPU | VRAM GB: 6.088359936
Selected examples: 1400 | training presentations: 1600


In [13]:
import sys
from pathlib import Path

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').exists()),
    Path.cwd(),
)
sys.path.insert(0, str(ROOT))

from router.metrics import load_splits
import pandas as pd

splits = load_splits()
cand = pd.read_parquet(ROOT / 'data/raw/llm_candidates/train-000.parquet')
# Candidate option pool: model id -> short description (used as criteria text).
MODEL_POOL = {r['model_name']: f"{r['model_name']} ({r['size']})" for _, r in cand.iterrows()}
assert len(MODEL_POOL) >= 2

rng = random.Random(SEED)
train = rng.sample(splits['train'], min(TRAIN_SIZE, len(splits['train'])))
valid = rng.sample(splits['valid'], min(TEST_SIZE, len(splits['valid'])))
test = rng.sample(splits['test'], min(TEST_SIZE, len(splits['test'])))
print('pool models:', len(MODEL_POOL))
print('train', len(train), '| valid', len(valid), '| test', len(test))

pool models: 18
train 1200 | valid 200 | test 200


In [14]:
def request(query, pool=None):
    pool = MODEL_POOL if pool is None else pool
    if not isinstance(query, str) or not query.strip():
        raise ValueError('A nonempty query is required.')
    if len(pool) < 2:
        raise ValueError('At least two options are required.')
    return {'state': query, 'questions': {
        'model': {'type': 'choice',
                  'question': 'Which candidate model should answer this query?',
                  'criteria': dict(pool)}
    }}

def clipped_state(tokenizer, text, max_state_tokens=MAX_STATE_TOKENS):
    ids = tokenizer.encode(text, add_special_tokens=False)[:max_state_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True)

def chat_messages(text):
    return [
        {'role': 'system', 'content': 'Choose one candidate model to answer the query. '
         'Return only a JSON object with a model key. Allowed model IDs and meanings: '
         + json.dumps(MODEL_POOL)},
        {'role': 'user', 'content': clipped_state(tok, text)},
    ]

def parse(reply):
    try:
        obj = json.loads(reply.strip())
    except (ValueError, TypeError):
        return None
    model = obj.get('model') if isinstance(obj, dict) else None
    return model if isinstance(model, str) and model in MODEL_POOL else None

print(json.dumps(request(train[0]['query']), indent=2))

{
  "state": "[System Instruction]\nAnswer the question using the provided context.\n\n[User Query]\nQuestion: To what should the language be referred to instead of \"Bosnian\"?\n\nContext: In Serbia, the Serbian language is the official one, while both Serbian and Croatian are official in the province of Vojvodina. A large Bosniak minority is present in the southwest region of Sand\u017eak, but the \"official recognition\" of Bosnian language is moot. Bosnian is an optional course in 1st and 2nd grade of the elementary school, while it is also in official use in the municipality of Novi Pazar. However, its nomenclature is controversial, as there is incentive that it is referred to as \"Bosniak\" (bo\u0161nja\u010dki) rather than \"Bosnian\" (bosanski) (see Bosnian language for details).",
  "questions": {
    "model": {
      "type": "choice",
      "question": "Which candidate model should answer this query?",
      "criteria": {
        "qwen2.5-7b-instruct": "qwen2.5-7b-instruct (7

In [15]:
print(f"[status] Loading tokenizer ... ({time.strftime('%H:%M:%S')})")
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL, revision=MODEL_REVISION, use_fast=True, trust_remote_code=False)
if not tok.is_fast:
    raise RuntimeError('A fast tokenizer is required for reliable character-to-token spans.')
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
assert tok.pad_token_id is not None
print(f"[status] Tokenizer loaded in {time.perf_counter()-t0:.1f}s")
print('[status] Loading model ...')
t0 = time.perf_counter()
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL, revision=MODEL_REVISION, dtype=DTYPE, attn_implementation='eager', trust_remote_code=False,
)
print(f"[status] Model loaded in {time.perf_counter()-t0:.1f}s; moving to {DEVICE} ...")
model = model.to(DEVICE).eval()
print(f"[status] Model on {DEVICE}")

[status] Loading tokenizer ... (14:24:58)
[status] Tokenizer loaded in 1.9s
[status] Loading model ...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 6866.60it/s]


[status] Model loaded in 0.5s; moving to cuda ...
[status] Model on cuda


In [16]:
@torch.no_grad()
def ask(texts):
    tok.padding_side = 'left'
    prompts = [tok.apply_chat_template(chat_messages(text), add_generation_prompt=True,
                                      tokenize=False, enable_thinking=False) for text in texts]
    batch = tok(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(DEVICE)
    out = model.generate(**batch, max_new_tokens=32, do_sample=False,
                         pad_token_id=tok.pad_token_id, use_cache=True)
    return [tok.decode(row[batch.input_ids.shape[1]:], skip_special_tokens=True) for row in out]

def evaluate_chat(sample, batch=EVAL_BATCH):
    preds = []
    for offset in tqdm(range(0, len(sample), batch), desc='Generative baseline'):
        part = sample[offset:offset + batch]
        preds.extend(parse(r) for r in ask([row['query'] for row in part]))
    n = len(sample)
    oracle_acc = sum(p == r['oracle_target'] for p, r in zip(preds, sample)) / n
    hit = sum(p in r['acceptable_models'] for p, r in zip(preds, sample) if p) / n
    invalid = preds.count(None)
    return oracle_acc, hit, invalid

base_oracle = base_hit = base_invalid = None
if RUN_CHAT_BASELINE:
    synchronize(); start = time.perf_counter()
    base_oracle, base_hit, base_invalid = evaluate_chat(test)
    synchronize(); base_ms = 1000 * (time.perf_counter() - start) / len(test)
    print({'chat_oracle_acc': base_oracle, 'chat_acceptable_hit': base_hit,
          'invalid_json': base_invalid, 'batched_ms_per_query': base_ms})
else:
    print('Generative baseline skipped.')

Generative baseline:   0%|          | 0/100 [00:00<?, ?it/s][transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
Generative baselin

{'chat_oracle_acc': 0.0, 'chat_acceptable_hit': 0.02, 'invalid_json': 192, 'batched_ms_per_query': 731.3640350149944}


In [17]:
@dataclass
class EncodedQuestion:
    key: str
    question_span: tuple
    option_ids: list
    option_spans: list

@dataclass
class EncodedRecord:
    input_ids: list
    state_span: tuple
    questions: list

def token_span(offsets, start, end):
    positions = [i for i, (left, right) in enumerate(offsets)
                 if right > left and right > start and left < end]
    if not positions:
        raise ValueError('A schema/state span has no tokens.')
    return positions[0], positions[-1] + 1

def encode_record(tokenizer, record, max_state_tokens=MAX_STATE_TOKENS, max_length=MAX_LENGTH):
    questions = record['questions']
    if len(questions) != 1:
        raise ValueError('This implementation supports exactly one choice question per request.')
    key, spec = next(iter(questions.items()))
    criteria = spec['criteria']
    if spec['type'] != 'choice' or len(criteria) < 2:
        raise ValueError('Expected a choice question with at least two criteria.')
    if not isinstance(spec['question'], str) or not spec['question'].strip():
        raise ValueError('Question text must be nonempty.')
    if any(not isinstance(k, str) or not isinstance(v, str) or not v.strip() for k, v in criteria.items()):
        raise ValueError('Option IDs and nonempty descriptions must be strings.')
    state = clipped_state(tokenizer, record['state'], max_state_tokens)
    if not state.strip():
        raise ValueError('State is empty after tokenization.')
    payload = 'STATE:\n' + state
    state_chars = (len('STATE:\n'), len(payload))
    payload += '\n\nSCHEMA:\nQuestion: '
    question_start = len(payload)
    payload += spec['question']
    question_chars = (question_start, len(payload))
    option_chars = []
    for option_id, description in criteria.items():
        payload += '\nOption: '
        start = len(payload)
        payload += option_id + ' — ' + description
        option_chars.append((start, len(payload)))
    prompt = tokenizer.apply_chat_template([
        {'role': 'system', 'content': 'Read the state and the schema. Compare the allowed options.'},
        {'role': 'user', 'content': payload},
    ], add_generation_prompt=True, tokenize=False, enable_thinking=False)
    prompt += 'JOINT SCHEMA DECISIONS:\n'
    if prompt.count(payload) != 1:
        raise ValueError('Chat template did not preserve the schema text exactly once.')
    origin = prompt.index(payload)
    encoded = tokenizer(prompt, add_special_tokens=False, return_offsets_mapping=True)
    ids, offsets = encoded['input_ids'], encoded['offset_mapping']
    if len(ids) > max_length:
        raise ValueError(f'Request needs {len(ids)} tokens; shorten the state/schema or increase max_length.')
    def span(chars):
        return token_span(offsets, origin + chars[0], origin + chars[1])
    return EncodedRecord(ids, span(state_chars), [EncodedQuestion(
        key, span(question_chars), list(criteria), [span(c) for c in option_chars]
    )])

def collate(encs, pad_token_id, device):
    if not encs:
        raise ValueError('Cannot collate an empty batch.')
    size, length = len(encs), max(len(e.input_ids) for e in encs)
    slots = max(1 + len(e.questions[0].option_ids) for e in encs)
    ids = torch.full((size, length), pad_token_id, dtype=torch.long)
    attention = torch.zeros(size, length, dtype=torch.long)
    state_mask = torch.zeros(size, length, dtype=torch.bool)
    pool = torch.zeros(size, slots, length, dtype=torch.float32)
    slot_mask = torch.zeros(size, slots, dtype=torch.bool)
    for i, enc in enumerate(encs):
        n = len(enc.input_ids)
        ids[i, :n] = torch.tensor(enc.input_ids)
        attention[i, :n] = 1
        a, b = enc.state_span
        state_mask[i, a:b] = True
        q = enc.questions[0]
        spans = [q.question_span, *q.option_spans]
        for j, (a, b) in enumerate(spans):
            pool[i, j, a:b] = 1.0 / (b - a)
            slot_mask[i, j] = True
    t = {'input_ids': ids, 'attention_mask': attention, 'state_mask': state_mask,
         'pool': pool, 'slot_mask': slot_mask}
    return {**{name: value.to(device) for name, value in t.items()}, 'encs': encs}

enc = encode_record(tok, request(train[0]['query']))
q = enc.questions[0]
print(len(enc.input_ids), 'tokens')
print('question:', tok.decode(enc.input_ids[slice(*q.question_span)]))
for oid, span in zip(q.option_ids[:3], q.option_spans[:3]):
    print(f'{oid:24}', tok.decode(enc.input_ids[slice(*span)]))
for row in train + test:
    encode_record(tok, request(row['query']))

703 tokens
question:  Which candidate model should answer this query?
qwen2.5-7b-instruct       qwen2.5-7b-instruct — qwen2.5-7b-instruct (7B)
gemma-2-9b-it             gemma-2-9b-it — gemma-2-9b-it (9B)
llama-3-8b-instruct-lite  llama-3-8b-instruct-lite — llama-3-8b-instruct-lite (8B)


In [18]:
class ChoiceHead(nn.Module):
    def __init__(self, hidden_size, width=128, num_heads=4, joint_layers=2, dropout=0.1):
        super().__init__()
        self.context = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, width))
        self.lexical = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, width))
        self.memory = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, width))
        self.roles = nn.Embedding(2, width)
        self.evidence = nn.MultiheadAttention(width, num_heads, dropout=dropout, batch_first=True)
        self.evidence_norm = nn.LayerNorm(width)
        self.joint = nn.ModuleList([
            nn.TransformerEncoderLayer(width, num_heads, dim_feedforward=2 * width,
                                       dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
            for _ in range(joint_layers)
        ])
        self.score = nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, width),
                                   nn.GELU(), nn.Linear(width, 1))

    def forward(self, hidden, lexical, batch):
        hidden, lexical = hidden.float(), lexical.float()
        pool = batch['pool']
        schema = self.context(pool @ hidden) + self.lexical(pool @ lexical)
        roles = torch.ones(schema.shape[1], dtype=torch.long, device=schema.device)
        roles[0] = 0
        schema = schema + self.roles(roles)[None]
        memory = self.memory(hidden)
        evidence, _ = self.evidence(schema, memory, memory,
                                    key_padding_mask=~batch['state_mask'], need_weights=False)
        schema = self.evidence_norm(schema + evidence)
        for layer in self.joint:
            schema = layer(schema, src_key_padding_mask=~batch['slot_mask'])
        question = schema[:, :1].expand(-1, schema.shape[1] - 1, -1)
        scores = self.score(torch.cat([schema[:, 1:], question], dim=-1)).squeeze(-1)
        return [[scores[i, :len(enc.questions[0].option_ids)]] for i, enc in enumerate(batch['encs'])]

class SystemOneRouter(nn.Module):
    def __init__(self, model, head, encoding_config=None):
        super().__init__()
        self.model = model
        self.head = head
        self.encoding_config = encoding_config or {'max_state_tokens': MAX_STATE_TOKENS, 'max_length': MAX_LENGTH}

    def forward(self, batch):
        base = self.model.get_base_model() if isinstance(self.model, PeftModel) else self.model
        text_model = base.model.language_model
        device_type = batch['input_ids'].device.type
        dtype = next(text_model.parameters()).dtype
        amp = torch.autocast('cuda', dtype=dtype) if device_type == 'cuda' and dtype in (torch.float16, torch.bfloat16) else nullcontext()
        with amp:
            hidden = text_model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
                                use_cache=False, return_dict=True).last_hidden_state
        lexical = text_model.get_input_embeddings()(batch['input_ids'])
        return self.head(hidden, lexical, batch)

head = ChoiceHead(model.config.text_config.hidden_size, **HEAD_CONFIG).to(DEVICE)
print('Head parameters:', sum(p.numel() for p in head.parameters()))

Head parameters: 764801


In [19]:
model.requires_grad_(False)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()
LORA = [name for name, layer in model.named_modules()
        if '.language_model.' in f'.{name}.' and isinstance(layer, nn.Linear)]
if not LORA:
    raise RuntimeError('No text-backbone linear layers found; check the Qwen implementation.')
assert not any('visual' in name or 'lm_head' in name for name in LORA)
print('LoRA target count:', len(LORA))
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                       target_modules=LORA, bias='none'))
clef = SystemOneRouter(model, head)
model.print_trainable_parameters()

LoRA target count: 186
trainable params: 10,822,656 || all params: 863,808,576 || trainable%: 1.2529


In [20]:
def train_request(row):
    others = [k for k in MODEL_POOL if k != row['oracle_target']]
    keep = list(MODEL_POOL)
    if rng.random() < 0.5:
        keep = [row['oracle_target'], *rng.sample(others, rng.randint(1, len(others)))]
    rng.shuffle(keep)
    return request(row['query'], {k: MODEL_POOL[k] for k in keep})

print(json.dumps(train_request(train[0])['questions']['model']['criteria'], indent=2))
# Variable-option shape/probability smoke check before the optimizer runs.
smoke_requests = [request(train[0]['query'], dict(list(MODEL_POOL.items())[:2])),
                  request(train[1]['query'])]
smoke_encs = [encode_record(tok, r) for r in smoke_requests]
smoke_batch = collate(smoke_encs, tok.pad_token_id, DEVICE)
assert torch.allclose(smoke_batch['pool'].sum(-1), smoke_batch['slot_mask'].float())
clef.eval()
with torch.no_grad():
    smoke_logits = clef(smoke_batch)
assert [len(lg[0]) for lg in smoke_logits] == [2, len(MODEL_POOL)]
assert all(torch.isfinite(lg[0]).all() for lg in smoke_logits)
del smoke_batch, smoke_logits

{
  "llama-3.3-70b-instruct-turbo": "llama-3.3-70b-instruct-turbo (70B)",
  "deepseek-v3.1": "deepseek-v3.1 (671B)",
  "llama-3-8b-instruct-lite": "llama-3-8b-instruct-lite (8B)",
  "qwen3-next-80b-a3b-instruct": "qwen3-next-80b-a3b-instruct (80B)",
  "gpt-oss-20b": "gpt-oss-20b (20B)",
  "gpt-oss-120b": "gpt-oss-120b (120B)",
  "gemma-2-9b-it": "gemma-2-9b-it (9B)"
}


In [21]:
opt = torch.optim.AdamW([
    {'params': [p for p in model.parameters() if p.requires_grad], 'lr': 2e-4},
    {'params': head.parameters(), 'lr': 1e-3},
], weight_decay=0.01)
WARMUP = min(20, max(1, STEPS // 5))

def lr_multiplier(step):
    if step < WARMUP:
        return (step + 1) / WARMUP
    progress = min(1.0, (step - WARMUP) / max(1, STEPS - WARMUP))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_multiplier)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda' and DTYPE == torch.float16))
trainable = [p for p in clef.parameters() if p.requires_grad]
rows = []
while len(rows) < STEPS * BATCH:
    order = list(train); rng.shuffle(order); rows.extend(order)
rows = rows[:STEPS * BATCH]
print('Distinct selected training queries:', len(train), '| presentations:', len(rows))

if DEVICE == 'cuda':
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
elif DEVICE == 'mps':
    torch.mps.empty_cache()
clef.train(); opt.zero_grad(set_to_none=True); synchronize()
start, losses = time.perf_counter(), []
for step in (bar := tqdm(range(STEPS), desc='Train decision model')):
    part = rows[step * BATCH:(step + 1) * BATCH]
    encs = [encode_record(tok, train_request(row)) for row in part]
    logits = clef(collate(encs, tok.pad_token_id, DEVICE))
    loss = torch.stack([
        F.cross_entropy(lg[0].float()[None], torch.tensor(
            [enc.questions[0].option_ids.index(row['oracle_target'])], device=DEVICE))
        for row, enc, lg in zip(part, encs, logits)
    ]).mean()
    if not torch.isfinite(loss):
        raise FloatingPointError('Non-finite loss; stop and inspect device/dtype support.')
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0, error_if_nonfinite=True)
    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    losses.append(loss.item())
    bar.set_postfix(loss=f'{sum(losses[-10:]) / len(losses[-10:]):.3f}')
synchronize(); train_secs = time.perf_counter() - start
print(f'{train_secs / 60:.2f} min')
if DEVICE == 'cuda':
    print(f'Peak CUDA tensor allocation: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')
elif DEVICE == 'mps':
    print(f'Current MPS driver allocation: {torch.mps.driver_allocated_memory() / 1e9:.2f} GB')

Distinct selected training queries: 1200 | presentations: 1600


Train decision model: 100%|██████████| 400/400 [19:23<00:00,  2.91s/it, loss=0.966]

19.39 min
Peak CUDA tensor allocation: 2.94 GB


In [22]:
@torch.no_grad()
def systemone(clef, tokenizer, record):
    clef.eval()
    device = next(clef.head.parameters()).device
    enc = encode_record(tokenizer, record, **clef.encoding_config)
    logits = clef(collate([enc], tokenizer.pad_token_id, device))[0]
    result = {}
    for q, scores in zip(enc.questions, logits):
        probs = scores.float().softmax(-1).cpu().tolist()
        best = max(range(len(probs)), key=probs.__getitem__)
        result[q.key] = {'type': 'choice', 'choice': q.option_ids[best],
                         'confidence': probs[best],
                         'probabilities': dict(zip(q.option_ids, probs))}
    return {'answers': result, 'usage': {'input_tokens': len(enc.input_ids), 'output_tokens': 0}}

example_request = request(train[0]['query'])
example = systemone(clef, tok, example_request)
assert abs(sum(example['answers']['model']['probabilities'].values()) - 1.0) < 1e-5
print(json.dumps(example, indent=2))

{
  "answers": {
    "model": {
      "type": "choice",
      "choice": "gpt-oss-20b",
      "confidence": 0.430073082447052,
      "probabilities": {
        "qwen2.5-7b-instruct": 0.021424343809485435,
        "gemma-2-9b-it": 0.30599653720855713,
        "llama-3-8b-instruct-lite": 0.09470999240875244,
        "qwen2.5-7b-instruct-turbo": 0.004521883558481932,
        "mistral-7b-instruct-v0.3": 0.005233072675764561,
        "qwen3-next-80b-a3b-instruct": 0.0035886410623788834,
        "llama3-70b-instruct": 0.004133682698011398,
        "mixtral-8x7b-instruct-v0.1": 0.003198223654180765,
        "mixtral-8x22b-instruct-v0.1": 0.0033385534770786762,
        "gpt-oss-20b": 0.430073082447052,
        "mistral-small-3-24b-instruct": 0.043566420674324036,
        "llama-4-maverick": 0.012343487702310085,
        "rnj-1-instruct": 0.03994195908308029,
        "gpt-oss-120b": 0.012751487083733082,
        "qwen3-coder-next": 0.003006965620443225,
        "deepseek-v3.1": 0.003361332230269

In [23]:
@torch.no_grad()
def evaluate(records=None, batch=EVAL_BATCH):
    records = test if records is None else records
    clef.eval()
    right = hit = 0
    preds = []
    for offset in tqdm(range(0, len(records), batch), desc='Decision evaluation'):
        part = records[offset:offset + batch]
        encs = [encode_record(tok, request(row['query'])) for row in part]
        for row, enc, lg in zip(part, encs, clef(collate(encs, tok.pad_token_id, DEVICE))):
            pred = enc.questions[0].option_ids[lg[0].argmax().item()]
            right += pred == row['oracle_target']
            hit += pred in row['acceptable_models']
            preds.append(pred)
    n = len(records)
    return right / n, hit / n, preds

synchronize(); start = time.perf_counter()
tuned_oracle, tuned_hit, tuned_preds = evaluate()
synchronize(); decision_ms = 1000 * (time.perf_counter() - start) / len(test)
comparison = {
    'test_examples': len(test),
    'generative_baseline': {'oracle_acc': base_oracle, 'acceptable_hit': base_hit,
                            'invalid_json': base_invalid},
    'trained_decision_model': {'oracle_acc': tuned_oracle, 'acceptable_hit': tuned_hit,
                              'output_tokens': 0, 'batched_ms_per_query': decision_ms},
    'training_seconds': train_secs,
}
print(json.dumps(comparison, indent=2))
print('Accuracy is measured, not guaranteed to improve on this tiny run.')

Decision evaluation: 100%|██████████| 100/100 [00:40<00:00,  2.49it/s]

{
  "test_examples": 200,
  "generative_baseline": {
    "oracle_acc": 0.0,
    "acceptable_hit": 0.02,
    "invalid_json": 192
  },
  "trained_decision_model": {
    "oracle_acc": 0.73,
    "acceptable_hit": 0.795,
    "output_tokens": 0,
    "batched_ms_per_query": 201.21894636497018
  },
  "training_seconds": 1163.1504314199992
}
Accuracy is measured, not guaranteed to improve on this tiny run.


In [24]:
def save_release(clef, tokenizer, directory=None):
    directory = Path(directory) if directory else Path('router_releases') / ('router_' + uuid.uuid4().hex[:10])
    directory.mkdir(parents=True, exist_ok=False)
    clef.model.save_pretrained(directory / 'adapter', safe_serialization=True, save_embedding_layers=False)
    tokenizer.save_pretrained(directory / 'tokenizer')
    torch.save({k: v.detach().cpu() for k, v in clef.head.state_dict().items()}, directory / 'head.pt')
    manifest = {'format': 'router-adapter-v1', 'model_id': MODEL, 'model_revision': MODEL_REVISION,
                'head_config': HEAD_CONFIG, 'encoding_config': clef.encoding_config,
                'model_pool': MODEL_POOL, 'loss': 'cross_entropy', 'calibrated': False,
                'train_size': len(train), 'test_size': len(test), 'steps': STEPS, 'batch': BATCH,
                'seed': SEED, 'metrics': comparison,
                'versions': {name: importlib.metadata.version(name) for name in ['torch', 'transformers', 'peft', 'datasets']}}
    (directory / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return directory.resolve()

def load_release(directory, device=DEVICE, dtype=DTYPE):
    directory = Path(directory)
    manifest = json.loads((directory / 'manifest.json').read_text())
    if manifest['format'] != 'router-adapter-v1':
        raise ValueError('Unexpected release format.')
    tokenizer = AutoTokenizer.from_pretrained(directory / 'tokenizer', use_fast=True)
    base = Qwen3_5ForConditionalGeneration.from_pretrained(
        manifest['model_id'], revision=manifest['model_revision'], dtype=dtype,
        attn_implementation='eager', trust_remote_code=False,
    )
    base.requires_grad_(False)
    adapted = PeftModel.from_pretrained(base, directory / 'adapter', is_trainable=False)
    restored_head = ChoiceHead(base.config.text_config.hidden_size, **manifest['head_config'])
    restored_head.load_state_dict(torch.load(directory / 'head.pt', map_location='cpu', weights_only=True), strict=True)
    restored = SystemOneRouter(adapted, restored_head, manifest['encoding_config']).to(device).eval()
    return restored, tokenizer, manifest

reference = systemone(clef, tok, example_request)
release_dir = save_release(clef, tok)
print('Saved:', release_dir)

clef.to('cpu'); gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
elif DEVICE == 'mps':
    torch.mps.empty_cache()
loaded_clef, loaded_tok, manifest = load_release(release_dir)
reloaded = systemone(loaded_clef, loaded_tok, example_request)
before = reference['answers']['model']['probabilities']
after = reloaded['answers']['model']['probabilities']
assert before.keys() == after.keys()
max_delta = max(abs(before[k] - after[k]) for k in before)
assert max_delta < 1e-3, f'Reload mismatch: {max_delta}'
print('Save/reload maximum probability difference:', max_delta)

Saved: /home/mr/Desktop/Fireworks/training/router_releases/router_8ac97088cc


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 3172.35it/s]


Save/reload maximum probability difference: 0.0
